# fix wigderson proofs

I'm leaving [sample wigderson proofs](./sample_wigderson_proofs.ipynb) unchanged, so that the test set is unaffected

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
from pathlib import Path
import typing as t
import re
from pprint import pprint
import coq_serapy as c
import json
import pickle
import random

from src.config import CONFIG
from src.coq_serapy_util import LemmaLocation, read_commands, get_section_name_from_command
from src.dataset import Example, COQ_WIGDERSON_DEV_SAMPLED_DATASET
from src.dataset.sample_wigderson_proofs import sample_wigderson_lemmas

In [ ]:
WIGDERSON_DIR = Path(CONFIG.ROOT_DIR) / "coq-projects/coq-wigderson"
WIGDERSON_DIR

## look at `remove_max_deg_adj'`

In [ ]:
TEST_FILE = WIGDERSON_DIR / "subgraph.v"
TEST_FILE

In [ ]:
TEST_FILE_COMMANDS = read_commands(TEST_FILE.read_text())

In [ ]:
num_lemmas = 0
for command in TEST_FILE_COMMANDS:
    try:
        lemma_name = c.coq_util.lemma_name_from_statement(command[0])
        if lemma_name.strip() == "":
            continue
        if "Theorem" not in command[0] and "Lemma" not in command[0]:
            continue
        num_lemmas += 1
        print(lemma_name, num_lemmas)
    except:
        continue

In [ ]:
# TODO: factor this into a utility in coq serapy utisl

section_regex = re.compile(r"Section\s+(\w+)")
end_section_regex = re.compile(r"End\s+(\w+)")
lemma_regex = re.compile(r"(Example|Lemma|Theorem)\s+([\w']+)\s*(\{\w+\})?\s*:([\s\S]*)\.")

section_names: t.List[str] = []
lemmas: t.List[LemmaLocation] = []
for command, line in TEST_FILE_COMMANDS:
    print("processing", command)
    
    section_match = section_regex.match(command)
    if section_match is not None:
        print("section match")
        section_name = section_match.group(1)
        section_names.append(section_name)
        continue

    end_section_match = end_section_regex.match(command)
    if end_section_match is not None:
        print("end section match")
        section_name = end_section_match.group(1)
        if section_name != section_names[-1]:
            raise ValueError(f"Section name mismatch: {section_name} != {section_names[-1]}")
        section_names.pop()
        continue

    try:
        lemma_name = c.coq_util.lemma_name_from_statement(command)
        if lemma_name.strip() == "":
            continue
        if "Theorem" not in command and "Lemma" not in command:
            continue
        print("lemma match", lemma_name)
        lemma_location = LemmaLocation(
            project_name="coq-wigderson",
            file_name=TEST_FILE.name,
            lemma_name=lemma_name,
            section_names=section_names.copy(),
            coq_version="8.13"
        )
        lemmas.append(lemma_location)
    except:
        pass

len(lemmas)


In [ ]:
lemmas

## filter out test set

In [ ]:
import pickle
W_TEST_SAMPLED_FILE = Path(CONFIG.ROOT_DIR) / "data/COQ_WIGDERSON_TEST_SAMPLED_DATASET.pkl"
print(W_TEST_SAMPLED_FILE)
# read the pickled file into TEST_SET
with open(W_TEST_SAMPLED_FILE, "rb") as f:
    TEST_SET = pickle.load(f)

len(TEST_SET)

In [ ]:
# test_set sorted by example.name
sorted(TEST_SET, key=lambda x: x.name)

## fix `remove_max_deg_adj'`

In [ ]:
REMOVE_MAX_DEG_ADJ_EXAMPLES = [
    example for example in TEST_SET if example.location.lemma_name == 'remove_max_deg_adj'
]
OTHER_EXAMPLES = [
    example for example in TEST_SET if example.location.lemma_name != 'remove_max_deg_adj'
]

REMOVE_MAX_DEG_ADJ_EXAMPLES

In [ ]:
PRIME_EXAMPLE = REMOVE_MAX_DEG_ADJ_EXAMPLES[1]
FIXED_PRIME_EXAMPLE = Example(
    location=LemmaLocation(
        project_name=PRIME_EXAMPLE.location.project_name,
        file_name=PRIME_EXAMPLE.location.file_name,
        lemma_name="remove_max_deg_adj'",
        section_names=PRIME_EXAMPLE.location.section_names,
        coq_version=PRIME_EXAMPLE.location.coq_version
    ),
    proposition_command=PRIME_EXAMPLE.proposition_command,
    gold_standard_proof=PRIME_EXAMPLE.gold_standard_proof,
)

TEST_SET_2 = OTHER_EXAMPLES + [REMOVE_MAX_DEG_ADJ_EXAMPLES[0]] + [FIXED_PRIME_EXAMPLE]

In [ ]:
# test_set sorted by example.name
sorted(TEST_SET_2, key=lambda x: x.name)

## filter out lemmas without proofs

In [ ]:
PROBLEMATIC_EXAMPLES = [
    example for example in TEST_SET_2
    if example.location.lemma_name == 'Sremove_elements' or example.location.lemma_name == 'Mremove_elements'
]
NON_PROBLEMATIC_EXAMPLES = [
    example for example in TEST_SET_2
    if example.location.lemma_name != 'Sremove_elements' and example.location.lemma_name != 'Mremove_elements'
]

PROBLEMATIC_EXAMPLES

In [ ]:
TEST_SET_3 = NON_PROBLEMATIC_EXAMPLES + [PROBLEMATIC_EXAMPLES[0]]
len(TEST_SET_3)

## sample 2 new lemmas

In [ ]:
EXCLUDE_DATASET = COQ_WIGDERSON_DEV_SAMPLED_DATASET + TEST_SET_3
len(EXCLUDE_DATASET)

In [ ]:
additional_lemmas = sample_wigderson_lemmas(2, EXCLUDE_DATASET)
additional_lemmas

In [ ]:
[example.name for example in additional_lemmas]

In [ ]:
sorted(TEST_SET_3, key=lambda x: x.name)

In [ ]:
sorted(COQ_WIGDERSON_DEV_SAMPLED_DATASET, key=lambda x: x.name)

## write updated set to file

In [ ]:
FINAL_TEST_SET = TEST_SET_3 + additional_lemmas
len(FINAL_TEST_SET)

In [ ]:
sorted(FINAL_TEST_SET, key=lambda x: x.name)

In [ ]:
# write the final test set to the original pickle file
with open(W_TEST_SAMPLED_FILE, "wb") as f:
    pickle.dump(FINAL_TEST_SET, f)

In [ ]:
# dump sorted example names as a csv
import csv

EXAMPLE_NAMES = sorted([example.name for example in FINAL_TEST_SET])

W_TEST_SAMPLED_CSV_FILE = Path(CONFIG.ROOT_DIR) / "data/COQ_WIGDERSON_TEST_SAMPLED_DATASET.csv"
with W_TEST_SAMPLED_CSV_FILE.open("w") as f:
    writer = csv.writer(f)
    writer.writerow(["example_name"])
    for example_name in EXAMPLE_NAMES:
        writer.writerow([
            example_name
        ])